In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.getcwd().endswith("/quotaclimat"):
    os.chdir("../../../..")

repo_root_path = os.path.abspath(os.path.dirname(os.getcwd()))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)
repo_root_path



'/root/Workspace'

In [2]:
from quotaclimat.data_ingestion.advertising_detection.e01_download_audio import (
    download_audio,
)
from quotaclimat.data_ingestion.advertising_detection.e02_create_chunks import (
    Chunk,
)
from quotaclimat.data_ingestion.advertising_detection.e04_group_chunks import (
    debug_pair,
)
from quotaclimat.data_ingestion.advertising_detection.processor import (
    chunk_grouping,
    chunk_creator,
)
from quotaclimat.data_ingestion.advertising_detection.tools.cache import LocalCache
from quotaclimat.data_ingestion.advertising_detection.e00_partition_window import (
    partition_week_program,
)
from quotaclimat.data_ingestion.advertising_detection.tools.visualizer.chunk_comparator import (
    generate_chunk_comparator,
)

import json
from datetime import timedelta

In [4]:
channel = "tf1"
start_date = "2025-05-05"

partition = partition_week_program(
    channel=channel,
    start_date=start_date,
    margin=timedelta(minutes=15),
)

with LocalCache(name="chunks", version="6f834f044fc78878") as chunk_cache:
    chunks: list[Chunk] = []
    for segment in partition:
        chunk_batch = json.loads(chunk_cache.get(segment.identifier + ".json"))
        chunks.extend([Chunk.from_dict(d) for d in chunk_batch])

def get_chunk_by_start(start_sec: float) -> Chunk | None:
    return next((c for c in chunks if abs(c.start_sec - start_sec) < 1), None)

def partition_of_chunk(chunk: Chunk):
    return next(
        (p for p in partition if p.start_date.timestamp() <= chunk.start_sec < p.end_date.timestamp()), None
    )

In [5]:
chunk_a = get_chunk_by_start(1746791307.59)
chunk_b = get_chunk_by_start(1746685307.72)

debug_pair(
    chunk_a,
    chunk_b,
    chunk_grouping,
)

partition_a = partition_of_chunk(chunk_a)
partition_b = partition_of_chunk(chunk_b)

html = generate_chunk_comparator(
    chunk_a=chunk_a,
    chunk_b=chunk_b,
    audio_a_path=(await download_audio(partition_a))[0],
    audio_b_path=(await download_audio(partition_b))[0],
    segment_a_start_epoch=partition_a.start_date.timestamp(),
    segment_b_start_epoch=partition_b.start_date.timestamp(),
    chunk_creator=chunk_creator,  # optionnel, utilise les params par défaut sinon
    padding_sec=1.0,  # configurable
)

with open(".cache/comparison.html", "w") as f:
    f.write(html)


DEBUG: chunk pair grouping analysis
  A: [1746791307.59s – 1746791310.16s]  channel=tf1
  B: [1746685307.72s – 1746685311.06s]  channel=tf1

[1] Minimum duration filter (>= 0.5 s)
    A duration: 2.577s  ✓
    B duration: 3.344s  ✓

[2] Acoustic pre-filter (_features_compatible)
    duration |A-B| = 0.766s  (tol=1.0)  ✓
    energy_mean rel_diff = 0.2037  (tol=0.1)  ✗  (A=0.0385, B=0.0307)
    spectral_centroid rel_diff = 0.0527  (tol=0.05)  ✗  (A=2203.8, B=2087.6)
    zcr_mean rel_diff = 0.1117  (tol=0.1)  ✗  (A=0.1176, B=0.1045)
  → BLOCKED: acoustic pre-filter rejected this pair.
